# Thread 彙整與 Thread-Ticker 匯出

將02階段匹配成功的留言-ticker彙整成thread-month-ticker

使用 `pipeline_runs.json` 指向的成功 02 階段結果。

## 處理規則

- 僅分批讀取已匹配的 comment Parquet。
- 使用既有 UTC datetime 與標準化 `thread_id`。
- 缺少 `thread_id` 的留言不參與 thread 彙整。
- 無效 timestamp 不納入 thread 起訖時間。
- `n_comments`：thread 內留言列數。
- `n_authors`：不同且非空白的 author 數；`[deleted]` 視為有效值。
- ticker 採跨檔案 distinct union。
- `parent_id` 僅供 audit。

## 月份定義

thread 的月份取自第一筆匹配留言的 UTC 月份。

同一 thread 後續出現的所有 ticker 都歸入該月份，因此這不是逐月 contemporaneous mention table。

## 輸出

產生：

- `threads.parquet`
- `thread_ticker.parquet`
- `run_summary.json`

不產生 stock pairs，也不新增任何研究篩選或排除條件。

唯一值統計使用 disk-backed SQLite。只有兩個輸出都成功驗證後才更新 `pipeline_runs.json`；每次執行建立新的 UTC 時間戳目錄，舊輸出保留。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, sqlite3, time, uuid
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())

import sys
sys.path.insert(0,str(ROOT/'Notebook'))
from pipeline_io import new_run, publish, input_run, remove_work_database

POINTER = ROOT/'Data/process/pipeline_runs.json'
MATCH_MANIFEST_PATH = None  # Set an explicit manifest path here to override the pipeline pointer.
if MATCH_MANIFEST_PATH is None:
    MATCH_MANIFEST_PATH = json.loads(POINTER.read_text())['matching_manifest']
MATCH_MANIFEST_PATH = Path(MATCH_MANIFEST_PATH)
MATCH_MANIFEST = json.loads(MATCH_MANIFEST_PATH.read_text())
assert MATCH_MANIFEST['status']=='complete' and MATCH_MANIFEST.get('schema_version')==2
MATCH_DIR = MATCH_MANIFEST_PATH.parent
FILES = sorted(MATCH_DIR.glob('*.comments.parquet'))
assert len(FILES)==len(MATCH_MANIFEST['input_metadata'])
BATCH = 50_000
OUT = new_run(ROOT,'thread')
SOURCE_METADATA = {str(p):[p.stat().st_size,p.stat().st_mtime_ns] for p in FILES}
REQUIRED = ['comment_id','parent_id','thread_id','author','created_datetime_utc','matched_tickers','matched_ticker_count']
for p in FILES: assert set(REQUIRED).issubset(pq.read_schema(p).names)
print('Matching manifest:',MATCH_MANIFEST_PATH)
print('Output:',OUT)
display(pd.DataFrame([{'file':p.name,'rows':pq.read_metadata(p).num_rows} for p in FILES]))


Matching manifest: C:\Users\eagle\RedditStock\Data\process\ticker_matching\20260922T032254Z_98f0008d\manifest.json
Output: C:\Users\eagle\RedditStock\Data\process\03_thread_construction\2026-09-22_03-38-31_UTC


,file,rows
0,wallstreetbets_comment_part1.comments.parquet,221516
1,wallstreetbets_comment_part2.comments.parquet,373356
2,wallstreetbets_comment_part3.comments.parquet,1437717
3,wallstreetbets_comment_part4.comments.parquet,2304903
4,wallstreetbets_comment_part5.comments.parquet,2808327
5,wallstreetbets_comment_part6.comments.parquet,1569148


In [2]:
THREAD_SCHEMA = pa.schema([('thread_id', pa.string()), ('first_timestamp', pa.timestamp('ns',tz='UTC')),
    ('last_timestamp', pa.timestamp('ns',tz='UTC')), ('n_comments', pa.int64()), ('n_authors', pa.int64()),
    ('tickers_in_thread', pa.list_(pa.string())), ('n_unique_tickers', pa.int64())])


In [3]:
WORK_DB = OUT/'_work/thread_aggregation.sqlite'
WORK_DB.parent.mkdir()
db = sqlite3.connect(WORK_DB)
db.execute('PRAGMA cache_size=-32768')
db.execute('PRAGMA temp_store=FILE')
db.executescript("""
CREATE TABLE threads(thread_id TEXT PRIMARY KEY, first_ns INTEGER, last_ns INTEGER, n_comments INTEGER) WITHOUT ROWID;
CREATE TABLE thread_authors(thread_id TEXT, author TEXT, PRIMARY KEY(thread_id,author)) WITHOUT ROWID;
CREATE TABLE thread_tickers(thread_id TEXT, ticker TEXT, PRIMARY KEY(thread_id,ticker)) WITHOUT ROWID;
CREATE TABLE file_authors(file TEXT, author TEXT, PRIMARY KEY(file,author)) WITHOUT ROWID;
CREATE TABLE file_threads(file TEXT, thread_id TEXT, PRIMARY KEY(file,thread_id)) WITHOUT ROWID;
""")

def aggregate(frame, file):
    db.executemany('INSERT OR IGNORE INTO file_authors VALUES (?,?)', ((file,a) for a in frame.author.unique() if a != ''))
    valid = frame.loc[frame.thread_id.notna()]
    db.executemany('INSERT OR IGNORE INTO file_threads VALUES (?,?)', ((file,t) for t in valid.thread_id.unique()))
    grouped = valid.groupby('thread_id', sort=False).agg(first=('created_datetime_utc','min'), last=('created_datetime_utc','max'), n=('comment_id','size'))
    rows = ((t, None if pd.isna(f) else f.value, None if pd.isna(l) else l.value, int(k)) for t,f,l,k in grouped.itertuples())
    db.executemany("""INSERT INTO threads VALUES (?,?,?,?) ON CONFLICT(thread_id) DO UPDATE SET
    first_ns = CASE WHEN excluded.first_ns IS NULL THEN threads.first_ns WHEN threads.first_ns IS NULL THEN excluded.first_ns ELSE MIN(threads.first_ns,excluded.first_ns) END,
    last_ns = CASE WHEN excluded.last_ns IS NULL THEN threads.last_ns WHEN threads.last_ns IS NULL THEN excluded.last_ns ELSE MAX(threads.last_ns,excluded.last_ns) END,
    n_comments = threads.n_comments + excluded.n_comments""", rows)
    authors = valid.loc[valid.author.ne(''), ['thread_id','author']].drop_duplicates()
    db.executemany('INSERT OR IGNORE INTO thread_authors VALUES (?,?)', authors.itertuples(index=False,name=None))
    pairs = valid[['thread_id','matched_tickers']].explode('matched_tickers').drop_duplicates()
    db.executemany('INSERT OR IGNORE INTO thread_tickers VALUES (?,?)', pairs.itertuples(index=False,name=None))
    db.commit()

def diagnostics(frame):
    dates = frame.created_datetime_utc
    return {'row_count':len(frame), 'date_min':None if dates.dropna().empty else dates.min().isoformat(),
            'date_max':None if dates.dropna().empty else dates.max().isoformat(),
            'unique_authors':int(frame.loc[frame.author.ne(''),'author'].nunique()),
            'unique_threads':int(frame.thread_id.nunique()),
            'missing_thread_count':int(frame.thread_id.isna().sum()),
            'invalid_timestamp_count':int(dates.isna().sum()),
            'missing_parent_id_count':int(frame.parent_id.eq('').sum()),
            'unusual_thread_format_count':int((frame.thread_id.notna() & ~frame.thread_id.fillna('').str.fullmatch('[a-z0-9]+')).sum())}


## 彙整 6 份 Enriched Comment 檔

每個 batch 保留：

- row 診斷
- date 診斷
- author 診斷
- thread 診斷

原始 comment Parquet 不會被重寫。

In [4]:
started = time.perf_counter()
summaries = []
for source in FILES:
    file_started = time.perf_counter()
    csv_name = source.name.replace('.comments.parquet','.csv')
    totals = {'file':csv_name, 'row_count':0, 'missing_thread_count':0, 'invalid_timestamp_count':0,
              'missing_parent_id_count':0, 'unusual_thread_format_count':0, 'date_min':None, 'date_max':None}
    for batch_no, batch in enumerate(pq.ParquetFile(source).iter_batches(batch_size=BATCH,columns=REQUIRED),1):
            frame = batch.to_pandas()
            assert frame.matched_ticker_count.ge(1).all()
            aggregate(frame, csv_name)
            diag = diagnostics(frame)
            for key in ['row_count','missing_thread_count','invalid_timestamp_count','missing_parent_id_count','unusual_thread_format_count']:
                totals[key] += diag[key]
            for key, op in [('date_min',min),('date_max',max)]:
                values = [v for v in [totals[key],diag[key]] if v is not None]
                totals[key] = op(values) if values else None
            entry = {'file':csv_name, 'batch':batch_no, **diag}
            print(entry, flush=True)
            del frame, batch
    assert pq.read_metadata(source).num_rows == totals['row_count']
    totals['unique_authors'] = db.execute('SELECT COUNT(*) FROM file_authors WHERE file=?',(csv_name,)).fetchone()[0]
    totals['unique_threads'] = db.execute('SELECT COUNT(*) FROM file_threads WHERE file=?',(csv_name,)).fetchone()[0]
    totals['processing_seconds'] = time.perf_counter()-file_started
    summaries.append(totals)
    display(pd.DataFrame([totals]))



{'file': 'wallstreetbets_comment_part1.csv', 'batch': 1, 'row_count': 50000, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2015-06-18T14:32:13+00:00', 'unique_authors': 4601, 'unique_threads': 5196, 'missing_thread_count': 9, 'invalid_timestamp_count': 9, 'missing_parent_id_count': 9, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part1.csv', 'batch': 2, 'row_count': 50000, 'date_min': '2015-06-18T14:35:08+00:00', 'date_max': '2015-10-21T15:14:13+00:00', 'unique_authors': 4129, 'unique_threads': 4613, 'missing_thread_count': 1, 'invalid_timestamp_count': 1, 'missing_parent_id_count': 1, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part1.csv', 'batch': 3, 'row_count': 50000, 'date_min': '2015-10-21T15:14:32+00:00', 'date_max': '2016-01-21T07:30:53+00:00', 'unique_authors': 4597, 'unique_threads': 3931, 'missing_thread_count': 4, 'invalid_timestamp_count': 4, 'missing_parent_id_count': 4, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part1.csv', 'batch': 4, 'row_count': 50000, 'date_min': '2016-01-21T07:32:14+00:00', 'date_max': '2016-04-04T14:25:45+00:00', 'unique_authors': 5258, 'unique_threads': 4185, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part1.csv', 'batch': 5, 'row_count': 21516, 'date_min': '2016-04-04T14:29:22+00:00', 'date_max': '2016-04-25T12:28:09+00:00', 'unique_authors': 3071, 'unique_threads': 1879, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


...

{'file': 'wallstreetbets_comment_part6.csv', 'batch': 29, 'row_count': 50000, 'date_min': '2020-07-29T19:14:18+00:00', 'date_max': '2020-07-31T16:04:58+00:00', 'unique_authors': 10748, 'unique_threads': 1049, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part6.csv', 'batch': 30, 'row_count': 50000, 'date_min': '2020-07-31T16:05:00+00:00', 'date_max': '2020-08-03T14:43:49+00:00', 'unique_authors': 10310, 'unique_threads': 1141, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part6.csv', 'batch': 31, 'row_count': 50000, 'date_min': '2020-08-03T14:43:51+00:00', 'date_max': '2020-08-05T11:55:26+00:00', 'unique_authors': 10334, 'unique_threads': 1143, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


{'file': 'wallstreetbets_comment_part6.csv', 'batch': 32, 'row_count': 19148, 'date_min': '2020-08-05T11:55:27+00:00', 'date_max': '2020-08-06T01:41:18+00:00', 'unique_authors': 5926, 'unique_threads': 595, 'missing_thread_count': 0, 'invalid_timestamp_count': 0, 'missing_parent_id_count': 0, 'unusual_thread_format_count': 0}


,file,row_count,missing_thread_count,invalid_timestamp_count,missing_parent_id_count,unusual_thread_format_count,date_min,date_max,unique_authors,unique_threads,processing_seconds
0,wallstreetbets_comment_part6.csv,1569148,0,0,0,0,2020-06-01T00:00:04+00:00,2020-08-06T01:41:18+00:00,79292,22840,9.529878


## 分批匯出全域 Thread Table

使用排序後的 SQLite cursor，逐一串流每個 thread 的唯一 ticker，並整理為排序後的 ticker 清單。

- Thread table 僅由「成功匹配且具有非空白標準化 `thread_id`」的留言建立。
- 聚合過程不使用 `parent_id`。

In [5]:
query = """SELECT t.thread_id,t.first_ns,t.last_ns,t.n_comments,
    (SELECT COUNT(*) FROM thread_authors a WHERE a.thread_id=t.thread_id)
    FROM threads t ORDER BY t.thread_id"""
ticker_cursor = iter(db.execute('SELECT thread_id,ticker FROM thread_tickers ORDER BY thread_id,ticker'))
next_ticker = next(ticker_cursor,None)
thread_cursor = db.execute(query)
thread_dest = OUT/'threads.parquet'
thread_partial = thread_dest.with_suffix('.parquet.partial')
thread_rows = grouped_comments = 0
with pq.ParquetWriter(thread_partial, THREAD_SCHEMA, compression='zstd') as writer:
    while True:
        rows = thread_cursor.fetchmany(5000)
        if not rows: break
        records = []
        for thread, first, last, count, authors in rows:
            tickers = []
            while next_ticker is not None and next_ticker[0] == thread:
                tickers.append(next_ticker[1])
                next_ticker = next(ticker_cursor,None)
            assert tickers and (first is None or last >= first)
            records.append({'thread_id':thread,'first_timestamp':first,'last_timestamp':last,
                            'n_comments':count,'n_authors':authors,'tickers_in_thread':tickers,'n_unique_tickers':len(tickers)})
            grouped_comments += count
        writer.write_table(pa.Table.from_pylist(records, schema=THREAD_SCHEMA))
        thread_rows += len(records)
assert next_ticker is None
assert pq.read_metadata(thread_partial).num_rows == thread_rows
thread_partial.replace(thread_dest)
print({'thread_rows':thread_rows, 'represented_matched_comments':grouped_comments})


{'thread_rows': 258609, 'represented_matched_comments': 8714953}


## 最終診斷與一致性檢查

- Comment 總列數不得改變。
- 所有 thread 的 `n_comments` 加總，應等於 matched comments 減去缺少 `thread_id` 的列數。
- Author 與 thread 數量以全資料的唯一值聯集計算，不直接加總各檔案結果。
- 比對來源 CSV 的檔案大小與修改時間，確認原始資料未被變更。

In [6]:
overall = {'file':'ALL_FILES'}
for key in ['row_count','missing_thread_count','invalid_timestamp_count','missing_parent_id_count','unusual_thread_format_count']:
    overall[key] = sum(s[key] for s in summaries)
for key, op in [('date_min',min),('date_max',max)]:
    values = [s[key] for s in summaries if s[key] is not None]
    overall[key] = op(values) if values else None
overall['unique_authors'] = db.execute('SELECT COUNT(DISTINCT author) FROM file_authors').fetchone()[0]
overall['unique_threads'] = thread_rows
overall['processing_seconds'] = time.perf_counter()-started
assert grouped_comments == overall['row_count']-overall['missing_thread_count']
assert thread_rows == db.execute('SELECT COUNT(*) FROM threads').fetchone()[0]
assert overall['row_count'] == (MATCH_MANIFEST if 'overall' in MATCH_MANIFEST else json.loads((MATCH_DIR/'summary.json').read_text()))['overall']['matched_comment_count']
assert SOURCE_METADATA == {p:[Path(p).stat().st_size,Path(p).stat().st_mtime_ns] for p in SOURCE_METADATA}
table = pd.DataFrame(summaries+[overall])
display(table)
db.close()
remove_work_database(WORK_DB)
display(next(pq.ParquetFile(FILES[0]).iter_batches(batch_size=3)).to_pandas())
display(next(pq.ParquetFile(thread_dest).iter_batches(batch_size=3)).to_pandas())
print('Thread aggregation finished:', OUT)


,file,row_count,missing_thread_count,invalid_timestamp_count,missing_parent_id_count,unusual_thread_format_count,date_min,date_max,unique_authors,unique_threads,processing_seconds
0,wallstreetbets_comment_part1.csv,221516,14,14,14,0,2012-04-11T16:46:43+00:00,2016-04-25T12:28:09+00:00,14439,19409,2.306682
1,wallstreetbets_comment_part2.csv,373356,0,0,0,0,2016-04-25T12:29:23+00:00,2017-04-01T06:13:54+00:00,25508,31185,3.819426
2,wallstreetbets_comment_part3.csv,1437717,0,0,0,0,2017-04-01T05:41:35+00:00,2018-12-31T00:05:03+00:00,77224,92081,13.182367
3,wallstreetbets_comment_part4.csv,2304903,0,0,0,0,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,116851,45685,15.483090
4,wallstreetbets_comment_part5.csv,2808327,0,0,0,0,2020-02-01T00:00:02+00:00,2020-06-01T00:00:07+00:00,128468,49828,19.053757
5,wallstreetbets_comment_part6.csv,1569148,0,0,0,0,2020-06-01T00:00:04+00:00,2020-08-06T01:41:18+00:00,79292,22840,9.529878
6,ALL_FILES,8714967,14,14,14,0,2012-04-11T16:46:43+00:00,2020-08-06T01:41:18+00:00,309073,258609,69.279034


,comment_id,author,created_utc,link_id,body,matched_tickers,matched_ticker_count,source_file,source_row,id,parent_id,thread_id,created_datetime_utc,created_date,year_month
0,c4b0pvu,secondhandsondek,1334162803.0,t3_s4jw1,This is a fantastic idea Ill toss mine up in a...,"[BIT, BMO, JPM]",3,wallstreetbets_comment_part1.csv,1,c4b0pvu,t3_s4jw1,s4jw1,2012-04-11 16:46:43+00:00,2012-04-11,2012-04
1,c4b1fpf,glorysk87,1334165948.0,t3_s4jw1,INTC is on 417,[INTC],1,wallstreetbets_comment_part1.csv,2,c4b1fpf,t3_s4jw1,s4jw1,2012-04-11 17:39:08+00:00,2012-04-11,2012-04
2,c4b1rmm,entsportsjunkie,1334167351.0,t3_s4jw1,straddle call straddle put put put straddle ca...,"[CALL, FOR, MORE, ONE, TECH]",5,wallstreetbets_comment_part1.csv,3,c4b1rmm,t3_s4jw1,s4jw1,2012-04-11 18:02:31+00:00,2012-04-11,2012-04


,thread_id,first_timestamp,last_timestamp,n_comments,n_authors,tickers_in_thread,n_unique_tickers
0,100w1h,2012-09-17 20:19:16+00:00,2012-10-09 21:26:37+00:00,5,4,"[FOR, GOOD, NOW, RUN, WELL]",5
1,10378k,2012-09-18 17:44:13+00:00,2012-10-08 22:12:49+00:00,10,9,"[AGO, ALL, AMD, ANY, APU, ATI, CAP, FOR, FULL,...",26
2,1056p3,2012-09-19 23:55:06+00:00,2012-10-04 16:47:59+00:00,15,5,"[ALL, ANY, APDN, ARE, FAST, FOR, GAIN, GOOD, G...",28


Thread aggregation finished: C:\Users\eagle\RedditStock\Data\process\03_thread_construction\2026-09-22_03-38-31_UTC


## 匯出 Thread-Ticker

將每個 thread 的唯一 ticker 指派至該 thread「首次匹配留言」所在的 UTC 月份。

- 若 thread 缺少首次匹配時間，停止匯出，不自行補月份。
- 輸出僅包含三欄：`thread_id`、`ticker`、`year_month`。
- 此表不包含 author 資訊。

In [7]:
RUN = OUT
SOURCE = OUT/'threads.parquet'
DEST = OUT/'thread_ticker.parquet'
PARTIAL = DEST.with_suffix('.parquet.partial')
SCHEMA = pa.schema([('thread_id',pa.string()),('thread_month',pa.string()),('ticker',pa.string())])
source_stat = (SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
started=time.perf_counter()
rows=threads=cross_month=0
month_counts={}
diagnostics=[]
previous_thread=None
all_tickers=set()  # bounded by the upstream ticker dictionary, not comment count
with pq.ParquetWriter(PARTIAL,SCHEMA,compression='zstd') as writer:
    for batch_no,batch in enumerate(pq.ParquetFile(SOURCE).iter_batches(batch_size=5000),1):
        frame=batch.to_pandas()
        assert frame.thread_id.notna().all() and frame.thread_id.ne('').all()
        assert frame.thread_id.is_monotonic_increasing and frame.thread_id.is_unique
        if previous_thread is not None: assert previous_thread < frame.thread_id.iloc[0]
        previous_thread=frame.thread_id.iloc[-1]
        first=pd.to_datetime(frame.first_timestamp,utc=True)
        last=pd.to_datetime(frame.last_timestamp,utc=True)
        assert first.notna().all(), 'Cannot assign thread month without a valid first timestamp'
        assert (last>=first).all()
        assert all(list(ts)==sorted(set(ts)) and len(ts)==count and count>0
                   for ts,count in zip(frame.tickers_in_thread,frame.n_unique_tickers))
        frame['thread_month']=first.dt.strftime('%Y-%m')
        cross_month+=int(frame.thread_month.ne(last.dt.strftime('%Y-%m')).sum())
        output=frame[['thread_id','thread_month','tickers_in_thread']].explode('tickers_in_thread').rename(columns={'tickers_in_thread':'ticker'})
        assert len(output)==int(frame.n_unique_tickers.sum())
        assert not output.duplicated().any()
        writer.write_table(pa.Table.from_pandas(output,schema=SCHEMA,preserve_index=False),row_group_size=50000)
        rows+=len(output)
        threads+=len(frame)
        all_tickers.update(output.ticker.unique())
        for month,count in output.groupby('thread_month').size().items():
            month_counts[month]=month_counts.get(month,0)+int(count)
        diag={'batch':batch_no,'input_thread_rows':len(frame),'output_rows':len(output),
              'date_min':first.min().isoformat(),'date_max':first.max().isoformat(),
              'unique_threads':int(output.thread_id.nunique()),'unique_tickers':int(output.ticker.nunique()),
              'unique_authors':'N/A: no author column'}
        diagnostics.append(diag)
        print(diag,flush=True)
assert pq.read_metadata(PARTIAL).num_rows==rows
assert threads==pq.read_metadata(SOURCE).num_rows
assert source_stat==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
PARTIAL.replace(DEST)
summary={'source':str(SOURCE),'output':str(DEST),'rows':rows,'unique_threads':threads,
         'unique_tickers':len(all_tickers),'unique_authors':'N/A: no author column',
         'date_min':min(d['date_min'] for d in diagnostics),'date_max':max(d['date_max'] for d in diagnostics),
         'thread_month_min':min(month_counts),'thread_month_max':max(month_counts),
         'threads_spanning_months':cross_month,'processing_seconds':time.perf_counter()-started,
         'thread_month_definition':'UTC month of first matched comment; all thread tickers assigned to that month',
         'validation':'unique sorted source threads, unique tickers per thread, exact explode counts and output footer checked'}
display(pd.DataFrame([summary]))
display(pd.DataFrame(sorted(month_counts.items()),columns=['thread_month','ticker_rows']))
display(next(pq.ParquetFile(DEST).iter_batches(batch_size=10)).to_pandas())

{'batch': 1, 'input_thread_rows': 5000, 'output_rows': 91371, 'date_min': '2012-09-17T20:19:16+00:00', 'date_max': '2015-08-11T02:19:13+00:00', 'unique_threads': 5000, 'unique_tickers': 1584, 'unique_authors': 'N/A: no author column'}


{'batch': 2, 'input_thread_rows': 5000, 'output_rows': 90707, 'date_min': '2015-06-27T15:32:41+00:00', 'date_max': '2016-02-01T08:45:52+00:00', 'unique_threads': 5000, 'unique_tickers': 1546, 'unique_authors': 'N/A: no author column'}


{'batch': 3, 'input_thread_rows': 5000, 'output_rows': 96517, 'date_min': '2015-11-05T13:50:23+00:00', 'date_max': '2016-07-19T19:29:06+00:00', 'unique_threads': 5000, 'unique_tickers': 1606, 'unique_authors': 'N/A: no author column'}


{'batch': 4, 'input_thread_rows': 5000, 'output_rows': 88912, 'date_min': '2016-02-25T17:49:11+00:00', 'date_max': '2016-05-20T17:20:26+00:00', 'unique_threads': 5000, 'unique_tickers': 1617, 'unique_authors': 'N/A: no author column'}


{'batch': 5, 'input_thread_rows': 5000, 'output_rows': 86029, 'date_min': '2016-05-09T16:51:37+00:00', 'date_max': '2017-02-11T23:37:18+00:00', 'unique_threads': 5000, 'unique_tickers': 1556, 'unique_authors': 'N/A: no author column'}


...

{'batch': 51, 'input_thread_rows': 5000, 'output_rows': 115406, 'date_min': '2020-07-12T05:37:57+00:00', 'date_max': '2020-07-31T02:35:33+00:00', 'unique_threads': 5000, 'unique_tickers': 2523, 'unique_authors': 'N/A: no author column'}


{'batch': 52, 'input_thread_rows': 3609, 'output_rows': 76314, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2020-08-06T01:21:55+00:00', 'unique_threads': 3609, 'unique_tickers': 2086, 'unique_authors': 'N/A: no author column'}


,source,output,rows,unique_threads,unique_tickers,unique_authors,date_min,date_max,thread_month_min,thread_month_max,threads_spanning_months,processing_seconds,thread_month_definition,validation
0,C:\Users\eagle\RedditStock\Data\process\03_thr...,C:\Users\eagle\RedditStock\Data\process\03_thr...,5146535,258609,5682,N/A: no author column,2012-04-11T16:46:43+00:00,2020-08-06T01:21:55+00:00,2012-04,2020-08,13148,5.919108,UTC month of first matched comment; all thread...,"unique sorted source threads, unique tickers p..."


,thread_month,ticker_rows
0,2012-04,359
1,2012-05,2439
2,2012-06,1336
3,2012-07,1269
4,2012-08,1450
...,...,...
96,2020-04,285991
97,2020-05,275796
98,2020-06,216889
99,2020-07,221440


,thread_id,thread_month,ticker
0,100w1h,2012-09,FOR
1,100w1h,2012-09,GOOD
2,100w1h,2012-09,NOW
3,100w1h,2012-09,RUN
4,100w1h,2012-09,WELL
5,10378k,2012-09,AGO
6,10378k,2012-09,ALL
7,10378k,2012-09,AMD
8,10378k,2012-09,ANY
9,10378k,2012-09,APU


In [8]:
pointer_state=json.loads(POINTER.read_text()) if POINTER.exists() else {}
assert pointer_state.get('matching_manifest')==str(MATCH_MANIFEST_PATH), 'Upstream pointer changed'
publish(ROOT,'thread',OUT,{'matching_input':str(MATCH_DIR),'batch_size':BATCH,
    'source_metadata':SOURCE_METADATA,'timezone':'UTC','files':summaries,'overall':overall,
    'thread_table_rows':thread_rows,'thread_table_n_comments_sum':grouped_comments,
    'thread_ticker':summary,'month_counts':month_counts,
    'missing_thread_policy':'retain comments; exclude from thread table',
    'invalid_date_policy':'retain comments; null derived dates'})
print('COMPLETE:',OUT)


COMPLETE: C:\Users\eagle\RedditStock\Data\process\03_thread_construction\2026-09-22_03-38-31_UTC
